# 🅿 ParkSense GridLock — YOLOv8x Computer Vision Pipeline

**Goal:** Document the PS3 computer vision pipeline that detects parking violations from
CCTV/traffic camera images and extracts license plates via EasyOCR.

**Source:** `backend/services/cv_detection.py` — production code on HuggingFace Spaces.

---


In [ ]:
# Install required packages (skip if already installed)
# !pip install ultralytics easyocr opencv-python-headless

import os
import uuid
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

AMBER = '#F5A623'; RED = '#FF3B5C'; TEAL = '#00D4AA'
BLUE  = '#4488FF'; MUTED = '#8B92A8'

plt.rcParams.update({
    'figure.facecolor': '#12172E', 'axes.facecolor': '#1A2040',
    'axes.edgecolor': '#505670',   'axes.labelcolor': '#F0F2F8',
    'xtick.color': '#8B92A8',      'ytick.color': '#8B92A8',
    'text.color': '#F0F2F8',       'grid.alpha': 0.05,
})
print('✓ Ready')

## 1. Architecture Overview

```
Traffic Camera Image
        │
        ▼
┌─────────────────────────────────────────┐
│  STAGE 1: YOLOv8x Object Detection      │
│  • Model : YOLOv8x (COCO pretrained)    │
│  • conf  = 0.35  │  IoU = 0.45          │
│  • Target COCO IDs: 2,3,5,7 (vehicles) │
└──────────────────┬──────────────────────┘
                   │  bounding boxes
                   ▼
┌─────────────────────────────────────────┐
│  STAGE 2: Violation Classification      │
│  • Spatial heuristics (context-based)  │
│  • Maps to 5 BTP violation categories  │
│  • Assigns CRITICAL / HIGH / MEDIUM     │
└──────────────────┬──────────────────────┘
                   │  cropped bbox (bottom 40%)
                   ▼
┌─────────────────────────────────────────┐
│  STAGE 3: EasyOCR Plate Extraction     │
│  • Allowlist: alphanumeric + space      │
│  • KA-format Karnataka plates           │
│  • Confidence threshold: >0.65          │
└──────────────────┬──────────────────────┘
                   │  JSON
                   ▼
    FastAPI /api/evidence/analyze
```


## 2. Model Configuration

In [ ]:
# Exact configuration from backend/services/cv_detection.py

# ── YOLOv8x Thresholds ──
CONFIDENCE_THRESHOLD = 0.35   # Below this = ignore detection
IOU_THRESHOLD        = 0.45   # Non-maximum suppression IoU threshold

# ── COCO class IDs mapped to vehicle types ──
COCO_VEHICLE_IDS = {
    2: 'car',
    3: 'motorcycle',
    5: 'bus',
    7: 'truck',
}
# Note: auto-rickshaw (common in Bengaluru) is not a COCO class.
# In production, fine-tuning adds class 80 = auto-rickshaw.

# ── Violation categories (BTP PS3 mapping) ──
VIOLATION_CLASSES = [
    'WRONG PARKING',
    'NO PARKING',
    'PARKING ON FOOTPATH',
    'PARKING NEAR ROAD CROSSING',
    'PARKING IN A MAIN ROAD',
]

# ── Severity tiers ──
VIOLATION_SEVERITY = {
    'PARKING NEAR ROAD CROSSING': 'CRITICAL',
    'PARKING ON FOOTPATH':        'CRITICAL',
    'PARKING IN A MAIN ROAD':     'HIGH',
    'WRONG PARKING':              'HIGH',
    'NO PARKING':                 'MEDIUM',
}

print('YOLOv8x config:')
print(f'  confidence threshold : {CONFIDENCE_THRESHOLD}')
print(f'  IoU threshold        : {IOU_THRESHOLD}')
print(f'  vehicle classes      : {list(COCO_VEHICLE_IDS.values())}')
print(f'  violation categories : {len(VIOLATION_CLASSES)}')

## 3. Why These Thresholds?

| Threshold | Value | Rationale |
|---|---|---|
| Confidence | **0.35** | Lower than default (0.25) to catch partial/occluded vehicles |
| IoU | **0.45** | Standard NMS — tighter than 0.7 prevents double-counting |
| Plate crop | **bottom 40%** | Karnataka plates always in lower portion of vehicle bbox |
| OCR min conf | **0.65** | Below this: mark as UNREAD, don't file false challan |

> **Key design choice:** False negatives (missing a violation) are preferred over false positives
> (wrongly charging a vehicle). Hence conf=0.35 but plate OCR confidence kept high at 0.65.


## 4. Demo Mode vs Production Mode

In [ ]:
# ParkSense runs in two modes (controlled by USE_REAL_MODEL env var)

print('┌──────────────────────────────────────────────────────────┐')
print('│               DEMO MODE (default on HF Spaces)           │')
print('├──────────────────────────────────────────────────────────┤')
print('│  USE_REAL_MODEL=false                                    │')
print('│  Returns realistic simulated detections                  │')
print('│  No GPU required · Instant response · No model weights  │')
print('│  Used for: hackathon demo, UI testing, API showcase      │')
print('└──────────────────────────────────────────────────────────┘')
print()
print('┌──────────────────────────────────────────────────────────┐')
print('│            PRODUCTION MODE (USE_REAL_MODEL=true)         │')
print('├──────────────────────────────────────────────────────────┤')
print('│  YOLOv8x COCO weights → vehicle detection                │')
print('│  EasyOCR → license plate OCR (EN, allowlist mode)       │')
print('│  Requires: ultralytics, easyocr, opencv, CUDA (opt)     │')
print('│  YOLO_WEIGHTS_PATH env var → custom weights path        │')
print('└──────────────────────────────────────────────────────────┘')

## 5. YOLOv8x — Model Capability Benchmark

In [ ]:
# YOLOv8x is the largest YOLOv8 variant
# Performance on COCO val2017:

model_comparison = {
    'Model':     ['YOLOv8n', 'YOLOv8s', 'YOLOv8m', 'YOLOv8l', 'YOLOv8x'],
    'mAP50-95':  [37.3,      44.9,      50.2,      52.9,      53.9],
    'Params(M)': [3.2,       11.2,      25.9,      43.7,      68.2],
    'Speed(ms)': [0.99,      1.20,      1.83,      2.39,      3.53],
}
import pandas as pd
df_models = pd.DataFrame(model_comparison)
df_models['Selected'] = df_models['Model'].apply(lambda x: '✅' if x == 'YOLOv8x' else '')
print(df_models.to_string(index=False))
print('\nRationale: mAP53.9 vs 52.9 (YOLOv8l) worth the extra latency')
print('HF Spaces CPU inference: ~3.5ms/img — acceptable for async challan generation')

In [ ]:
# Visualise the model size vs accuracy tradeoff
import matplotlib.pyplot as plt
import numpy as np

models    = ['YOLOv8n', 'YOLOv8s', 'YOLOv8m', 'YOLOv8l', 'YOLOv8x']
map_vals  = [37.3, 44.9, 50.2, 52.9, 53.9]
params    = [3.2, 11.2, 25.9, 43.7, 68.2]
colors    = [MUTED, MUTED, MUTED, MUTED, AMBER]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mAP comparison
bars = axes[0].bar(models, map_vals, color=colors, edgecolor='none')
axes[0].set_ylabel('mAP50-95 (COCO val2017)')
axes[0].set_title('YOLOv8 Variants — Detection Accuracy', fontsize=11)
axes[0].set_ylim(30, 58)
for bar, val in zip(bars, map_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.3,
                 f'{val}', ha='center', fontsize=10)
axes[0].text(4, 54.5, '← Selected', color=AMBER, ha='center', fontsize=10)

# Size vs accuracy scatter
for i, (m, p, a) in enumerate(zip(models, params, map_vals)):
    c = AMBER if m == 'YOLOv8x' else MUTED
    axes[1].scatter(p, a, s=120, c=c, zorder=5)
    axes[1].annotate(m, (p, a), textcoords='offset points',
                     xytext=(6, 4), fontsize=9, color=c)

axes[1].set_xlabel('Parameters (Millions)')
axes[1].set_ylabel('mAP50-95')
axes[1].set_title('Model Size vs Accuracy Tradeoff', fontsize=11)

plt.suptitle('YOLOv8 Model Selection — ParkSense GridLock CV Pipeline', fontsize=13)
plt.tight_layout()
plt.savefig('cv_01_model_selection.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Simulated Detection Output (Demo Mode)

In [ ]:
# Reproduce the demo detection logic from cv_detection.py
import uuid, random

random.seed(42)

VEHICLE_CLASSES = ['car', 'motorcycle', 'scooter', 'truck', 'bus', 'auto-rickshaw']
KA_SUFFIXES     = ['AA', 'BB', 'CC', 'MK', 'XX', 'HH', 'MM', 'PE', 'RE', 'HK']

def fake_plate():
    return f'KA{random.randint(1,99):02d} {random.choice(KA_SUFFIXES)}{random.randint(1000,9999)}'

def simulate_detection(filename='traffic_image.jpg'):
    n = random.randint(1, 4)
    detections = []
    for _ in range(n):
        viol = random.choice(VIOLATION_CLASSES)
        detections.append({
            'id': str(uuid.uuid4())[:8],
            'vehicle_class': random.choice(VEHICLE_CLASSES),
            'confidence': round(random.uniform(0.72, 0.97), 3),
            'violation': viol,
            'violation_confidence': round(random.uniform(0.68, 0.95), 3),
            'severity': VIOLATION_SEVERITY.get(viol, 'MEDIUM'),
            'bbox': {'x1': random.randint(40,180), 'y1': random.randint(40,120),
                     'x2': random.randint(280,580), 'y2': random.randint(220,420)},
            'license_plate': fake_plate(),
            'plate_confidence': round(random.uniform(0.65, 0.92), 3),
        })
    return {
        'evidence_id': str(uuid.uuid4()),
        'filename': filename,
        'model': 'YOLOv8x (COCO fine-tuned, BTP dataset) — DEMO MODE',
        'total_vehicles_detected': len(detections),
        'total_violations_detected': len(detections),
        'detections': detections,
        'status': 'processed',
    }

result = simulate_detection('Jayanagara_CCTV_0842.jpg')
import json
print(json.dumps(result, indent=2))

## 7. Bounding Box Visualisation

In [ ]:
# Visualise what a detection output looks like on a synthetic image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(7)
result = simulate_detection('Madiwala_Junction_1743.jpg')

SEVERITY_COLORS = {'CRITICAL': RED, 'HIGH': AMBER, 'MEDIUM': BLUE}

fig, ax = plt.subplots(1, 1, figsize=(11, 7))

# Draw synthetic road background
ax.set_facecolor('#2A3A2A')  # road surface
ax.add_patch(patches.Rectangle((0, 180), 640, 80, facecolor='#F5F5DC', alpha=0.3))  # road marking
ax.text(320, 340, 'Traffic Camera Frame — Madiwala Junction\n(Synthetic demo — real model uses live CCTV)',
        ha='center', va='center', fontsize=9, color='#8B92A8')

for i, det in enumerate(result['detections']):
    b = det['bbox']
    color = SEVERITY_COLORS.get(det['severity'], MUTED)
    # Offset boxes so they don't overlap in demo
    x1 = b['x1'] + i * 110
    rect = patches.Rectangle(
        (x1, b['y1']), b['x2']-b['x1'], b['y2']-b['y1'],
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    label = f"{det['vehicle_class'].upper()} {det['confidence']:.2f}\n{det['violation'][:18]}\n{det['license_plate']}"
    ax.text(x1, b['y1'] - 5, label, fontsize=7.5, color=color,
            va='bottom', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#12172E', alpha=0.8))

ax.set_xlim(0, 640)
ax.set_ylim(0, 480)
ax.set_title(f'YOLOv8x Detection — {len(result["detections"])} violations detected', fontsize=12)
ax.set_xlabel('Pixel X')
ax.set_ylabel('Pixel Y')

# Legend
legend_patches = [patches.Patch(color=c, label=t) for t, c in SEVERITY_COLORS.items()]
ax.legend(handles=legend_patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('cv_02_detection_output.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Detected: {result["total_violations_detected"]} violations')

## 8. EasyOCR License Plate Pipeline

In [ ]:
# EasyOCR approach from _real_inference() in cv_detection.py

print('License Plate Extraction Logic:')
print('─' * 50)
print()
print('1. Take vehicle bounding box from YOLO detection')
print('2. Crop BOTTOM 40% of bbox (plate always in lower portion)')
print('   plate_y1 = y1 + int((y2 - y1) * 0.6)')
print('   plate_crop = img[plate_y1:y2, x1:x2]')
print()
print('3. Run EasyOCR on cropped region:')
print('   reader = easyocr.Reader(["en"])')
print('   allowlist = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ "')
print()
print('4. Filter results by confidence >= 0.65')
print('5. Join text tokens → plate string')
print('6. If no result: plate = "UNREAD" → flagged for manual review')
print()
print('Karnataka plate format: KA[01-99] [AA-ZZ][0001-9999]')
print('  Example: KA01 MK1234  →  District 01 · Series MK · Number 1234')

In [ ]:
# Visualise plate extraction crop logic
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Simulate: full bbox, cropped region, OCR result
random.seed(3)
det = simulate_detection()['detections'][0]
b = det['bbox']
h, w = b['y2'] - b['y1'], b['x2'] - b['x1']

for ax, title, extra in zip(axes, 
    ['Full Vehicle BBox', 'Plate Crop (bottom 40%)', 'EasyOCR Result'],
    [None, None, det['license_plate']]):
    ax.set_facecolor('#2A3A2A')
    # Full vehicle
    ax.add_patch(patches.Rectangle((0.05, 0.1), 0.9, 0.85,
                 linewidth=2, edgecolor=AMBER, facecolor='#3A4A3A', alpha=0.4))
    ax.text(0.5, 0.55, '🚗', ha='center', va='center', fontsize=30)
    if title != 'Full Vehicle BBox':
        # Plate highlight
        ax.add_patch(patches.Rectangle((0.05, 0.1), 0.9, 0.34,
                     linewidth=2, edgecolor=TEAL, facecolor=TEAL, alpha=0.3))
        ax.text(0.5, 0.27, '🔢', ha='center', va='center', fontsize=22)
    if extra:
        ax.text(0.5, 0.05, extra, ha='center', va='bottom',
                fontsize=13, fontweight='bold', color=TEAL,
                bbox=dict(boxstyle='round', facecolor='#12172E', alpha=0.9))
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle('EasyOCR License Plate Extraction — Crop Strategy', fontsize=12, color='#F0F2F8')
plt.tight_layout()
plt.savefig('cv_03_ocr_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. FastAPI Integration — `/api/evidence/analyze`

```python
# backend/routes/evidence.py
from backend.services.cv_detection import analyze_image

@router.post('/analyze')
async def analyze_traffic_image(file: UploadFile):
    image_bytes = await file.read()
    result = analyze_image(image_bytes, file.filename)
    return result
```

**Live API endpoint:**  
`POST https://saritha-1-parksense-gridlock-api.hf.space/api/evidence/analyze`

**Sample cURL:**
```bash
curl -X POST 'https://saritha-1-parksense-gridlock-api.hf.space/api/evidence/analyze' \
     -F 'file=@parking_violation.jpg'
```


## 10. CV Pipeline Summary

| Component | Detail |
|---|---|
| Base model | YOLOv8x (68.2M params, mAP 53.9) |
| Confidence | 0.35 |
| IoU (NMS) | 0.45 |
| Vehicle classes | car, motorcycle, bus, truck (COCO) |
| Violation classes | 5 BTP categories |
| OCR engine | EasyOCR [EN], allowlist mode |
| Plate crop | bottom 40% of vehicle bbox |
| OCR min confidence | 0.65 |
| Deployment | HuggingFace Spaces (CPU, DEMO mode) |
| Production switch | `USE_REAL_MODEL=true` env var |

**Integration with PS1 pipeline:**  
CV detections feed back into the DBSCAN hotspot model — each confirmed CV violation
increments the violation count for the nearest cluster, updating enforcement priority in real time.
